# Cortex Search + RAG — Semantic Search & Retrieval-Augmented Generation

Combine **Cortex Search Service** (managed semantic search) with `COMPLETE` for RAG.

| Item | Detail |
|---|---|
| **Capabilities** | `EMBED_TEXT_1024`, `VECTOR_COSINE_SIMILARITY`, Cortex Search Service |
| **Data** | `wiki_articles` table (from AI_SUMMARIZE lab) |
| **Architecture** | Embed → Search → Generate (RAG pattern) |

> **Prerequisite:** Run the **AI_SUMMARIZE** lab first to create the `wiki_articles` table.


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Create Embeddings

Generate vector embeddings using `EMBED_TEXT_1024` for similarity search.

In [ ]:
CREATE OR REPLACE TABLE article_embeddings AS
SELECT
    article_id,
    title,
    category,
    body,
    SNOWFLAKE.CORTEX.EMBED_TEXT_1024('e5-base-v2', body) AS embedding
FROM wiki_articles;

In [ ]:
SELECT article_id, title, ARRAY_SIZE(embedding) AS vector_dims FROM article_embeddings;

## Step 3 — Manual Vector Similarity Search

Use `VECTOR_COSINE_SIMILARITY` for nearest-neighbor search.

In [ ]:
SELECT
    title,
    category,
    VECTOR_COSINE_SIMILARITY(
        embedding,
        SNOWFLAKE.CORTEX.EMBED_TEXT_1024('e5-base-v2', 'How do vector databases enable AI applications?')
    ) AS similarity
FROM article_embeddings
ORDER BY similarity DESC
LIMIT 3;

## Step 4 — Create Cortex Search Service

A managed service that handles indexing, embedding, and retrieval automatically.

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE wiki_search
    ON body
    ATTRIBUTES title, category
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 minute'
    AS (
        SELECT article_id, title, category, body
        FROM wiki_articles
    );

## Step 5 — RAG Pattern (Retrieve + Generate)

Retrieve relevant context with vector search, then pass to `COMPLETE` for grounded answers.

In [ ]:
WITH context AS (
    SELECT
        title,
        body,
        VECTOR_COSINE_SIMILARITY(
            embedding,
            SNOWFLAKE.CORTEX.EMBED_TEXT_1024('e5-base-v2', 'What is Snowflake and what AI features does it offer?')
        ) AS similarity
    FROM article_embeddings
    ORDER BY similarity DESC
    LIMIT 2
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'claude-3-5-haiku',
    'Based on the following context, answer the question: What is Snowflake and what AI features does it offer?\n\nContext:\n'
    || LISTAGG(title || ': ' || body, '\n\n') WITHIN GROUP (ORDER BY similarity DESC)
) AS rag_answer
FROM context;

## Key Takeaways

- **Embeddings** convert text to vectors for semantic (meaning-based) search.
- **Cortex Search Service** manages indexing and retrieval — no infrastructure needed.
- **RAG pattern**: Retrieve relevant docs → inject as context → generate grounded answer.
- Use `VECTOR_COSINE_SIMILARITY` for manual search, Cortex Search for managed service.
- The `TARGET_LAG` parameter controls how fresh the search index stays.
